In [ ]:
from collections import Counter
from pathlib import Path
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
from sympy import true

import part1 # to use code from tutorial
import torch

# Shared Step: Create Datasets with Three Representations

In [21]:
# splitting data into train and test sets and folds for cross-validation
LABEL_COL = "Party_orientation"
SAMPLE_FRAC = 0.2  # fraction of the data to use, 1.0 for all of it

# merge the party orientations into two classes, "Left" and "Right"
# "Centre" has very few samples, so it is discarded
orientation_to_side = {
    "Centre-left to left": "Left",
    "Centre-left": "Left",
    "Centre to centre-left": "Left",
    "Centre to centre-right": "Right",
    "Centre-right": "Right",
    "Right": "Right",
    "Right to far-right": "Right",
    "Far-right": "Right",
}

def merge_orientations(df, label_column=LABEL_COL):
    df = df[df[label_column].isin(orientation_to_side.keys())].copy()
    df[label_column] = df[label_column].map(orientation_to_side)
    return df

# get the dataframe: one dataframe with tokens, lemmas, sentences and embeddings,
# so all three representations are trained and evaluated on the same rows
df = merge_orientations(pd.read_pickle("./embeddings_sampled.pkl"))
df = df.sample(frac=SAMPLE_FRAC, random_state=42).reset_index(drop=True)

# train test split
train_df, test_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df[LABEL_COL])
train_df = train_df.reset_index(drop=True)

# labels and settings
label_names = sorted(df[LABEL_COL].unique())
outcome_space = part1.clslib.NamedClasses(label_names)
my_seeds = [0] # we can have more
num_folds = 5

In [22]:
# to save results
results_folder = Path("./results")
results_folder.mkdir(exist_ok=True)
# to save models
my_classifiers = {}

In [ ]:
def get_vocab(texts, min_count=5, max_size=None):
    counts = Counter(item for text in texts for item in text.split())
    kept = [item for item, count in counts.most_common(max_size) if count >= min_count]
    return {item: i for i, item in enumerate(kept)}

class Encoder:
    def __len__(self) -> int:
        raise NotImplementedError

    def __call__(self, x: str) -> np.ndarray:
        raise NotImplementedError

def sparse_row(counts: dict, size: int) -> csr_matrix:
    """A 1 x size sparse row with counts[index] at each index (only the non-zero entries are stored)."""
    indices = sorted(counts)
    values = np.array([counts[i] for i in indices], dtype=np.float32)
    return csr_matrix((values, indices, [0, len(indices)]), shape=(1, size))

class TokenIndicator(Encoder):
    def __init__(self, vocab: dict):
        self.vocab = vocab

    def __len__(self) -> int:
        return len(self.vocab)

    def __call__(self, x: str) -> csr_matrix:
        # sparse row instead of np.zeros(len(vocab)), so the full dataset fits in memory
        indices = {self.vocab[item] for item in x.split() if item in self.vocab}
        return sparse_row({index: 1 for index in indices}, len(self))

# Function to create pairs of texts and labels so we can use functions from part1 (tutorial)
def make_pairs(df, column, label_names, label_column="Party_orientation"):
    label_to_id = {name: i for i, name in enumerate(label_names)}
    texts = df[column]
    # only join list columns (tokens/lemmas); str.join on a plain string column would put spaces between characters
    if isinstance(texts.iloc[0], (list, tuple, np.ndarray)):
        texts = texts.str.join(" ")
    labels = [label_to_id[name] for name in df[label_column]]
    return list(zip(texts, labels))


In [24]:
# cross-validation folds from the train rows only (the test set stays held out), using the tutorial's function
def make_cv(column):
    pairs = make_pairs(train_df, column, label_names)
    return [part1.experiment.cross_validation_split(pairs=pairs, k=num_folds, seed=seed) for seed in my_seeds]

cv_lemmas = make_cv("lemmas")        # for token indicator and bag of words
cv_sentences = make_cv("sentence")   # for the embeddings

In [ ]:
# evaluation on the held-out test set: accuracy and macro F1
# (macro F1 averages the F1 of Left and Right, so the smaller Left class counts as much as Right)
from sklearn.metrics import accuracy_score, f1_score

test_lemmas = make_pairs(test_df, "lemmas", label_names)
test_sentences = make_pairs(test_df, "sentence", label_names)
test_scores = {}

def evaluate_on_test(name, test_pairs, encoder, model_option):
    """Loads every fold model of experiment `name`, predicts the test set and returns accuracy / macro F1 per model."""
    texts = [text for text, _ in test_pairs]
    gold = [label for _, label in test_pairs]
    rows = []
    for seed, run in zip(my_seeds, my_classifiers[name]):
        for fold, path in run["models"].items():
            if model_option == "logistic":
                preds = part1.loglin.predict(texts, part1.loglin.load(path), encoder)
            else:
                preds = part1.smlp.predict_probs(texts, part1.smlp.load(path), encoder, batch_size=512).argmax(axis=1)
            rows.append({"seed": seed, "fold": fold,
                         "accuracy": accuracy_score(gold, preds),
                         "f1": f1_score(gold, preds, average="macro")})
    scores = pd.DataFrame(rows).set_index(["seed", "fold"])
    test_scores[name] = scores
    print(scores.round(4).to_string())
    print(f"test accuracy {scores.accuracy.mean():.4f} ± {scores.accuracy.std():.4f} | "
          f"test macro F1 {scores.f1.mean():.4f} ± {scores.f1.std():.4f}")
    return scores

In [25]:
# vocabulary from the train rows only (lemmas)
train_texts = [text for text, _ in make_pairs(train_df, "lemmas", label_names)]
vocab = get_vocab(train_texts, min_count=5, max_size=5000)

In [ ]:
class BagOfWords(Encoder):
    def __init__(self, vocab: dict):
        self.vocab = vocab

    def __len__(self) -> int:
        return len(self.vocab)

    def __call__(self, x: str) -> csr_matrix:
        # repeated words are counted; sparse row so the full dataset fits in memory
        counts = Counter(self.vocab[item] for item in x.split() if item in self.vocab)
        return sparse_row(counts, len(self))

In [27]:
class Embedding(Encoder):
    dense = True  # tells part1 to use dense matrices (and lbfgs for logistic regression) instead of sparse ones

    def __init__(self, embeddings_map: pd.DataFrame):
        # build a sentence -> vector dict once, so each lookup is O(1) instead of scanning the whole dataframe
        self.lookup = {
            sentence: np.asarray(embedding, dtype=np.float32)
            for sentence, embedding in zip(embeddings_map['sentence'], embeddings_map['embeddings'])
        }
    def __len__(self) -> int:
        return 1024
    def __call__(self, x: str) -> np.ndarray:
        return self.lookup[x]

In [28]:
# encoders

token_ind_encoder = TokenIndicator(vocab)

bow_encoder = BagOfWords(vocab)

sentence_em_encoder = Embedding(df)

#print("vocabulary size:", len(token_ind_encoder))

# Experiment 1: Log-Lin

Exp 1.A: Log-Lin with Token Indicator

In [29]:
my_classifiers["logistic_indicator-lemmas"] = []

for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/logistic_indicator-lemmas/seed-{seed}"
    my_classifiers["logistic_indicator-lemmas"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=cv_lemmas[run],
            outcome_space=outcome_space,
            encoder=token_ind_encoder,
            model_option="logistic",
            save_to=run_folder
        )
    )

accuracies = my_classifiers["logistic_indicator-lemmas"][0]["accuracies"]

for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

  0%|          | 0/5 [00:00<?, ?it/s]

/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which 

fold 0: accuracy 0.6758
fold 1: accuracy 0.6788
fold 2: accuracy 0.6813
fold 3: accuracy 0.6771
fold 4: accuracy 0.6773


In [ ]:
evaluate_on_test("logistic_indicator-lemmas", test_lemmas, token_ind_encoder, "logistic");

Exp 1.B: Log-Lin with Bag of Words

In [30]:
my_classifiers["logistic_bow-lemmas"] = []

for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/logistic_bow-lemmas/seed-{seed}"
    my_classifiers["logistic_bow-lemmas"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=cv_lemmas[run],
            outcome_space=outcome_space,
            encoder=bow_encoder,
            model_option="logistic",
            save_to=run_folder
        )
    )

accuracies = my_classifiers["logistic_bow-lemmas"][0]["accuracies"]

for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

  0%|          | 0/5 [00:00<?, ?it/s]

/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/chrisfeltner/Git/nlp/nlp-project/.venv/lib/python3.15/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which 

fold 0: accuracy 0.6784
fold 1: accuracy 0.6797
fold 2: accuracy 0.6817
fold 3: accuracy 0.6747
fold 4: accuracy 0.6805


In [ ]:
evaluate_on_test("logistic_bow-lemmas", test_lemmas, bow_encoder, "logistic");

Exp 1.C: Log-Lin with Sentence Embeddings

In [31]:
my_classifiers["logistic_embeddings"] = []

for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/logistic_embeddings/seed-{seed}"
    my_classifiers["logistic_embeddings"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=cv_sentences[run],
            outcome_space=outcome_space,
            encoder=sentence_em_encoder,
            model_option="logistic",
            save_to=run_folder,
            loglin_args= {
                'max_iter': 300
            }
        )
    )

accuracies = my_classifiers["logistic_embeddings"][0]["accuracies"]

for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

  0%|          | 0/5 [00:00<?, ?it/s]

fold 0: accuracy 0.6503
fold 1: accuracy 0.6567
fold 2: accuracy 0.6538
fold 3: accuracy 0.6550
fold 4: accuracy 0.6548


In [ ]:
evaluate_on_test("logistic_embeddings", test_sentences, sentence_em_encoder, "logistic");

# Experiment 2: Multi-Layer Perceptron
One hidden layer MLP

Exp 2.A: MLP with Token Indicator

In [ ]:
my_classifiers["mlp_indicator-lemmas"] = []

for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/mlp_indicator-lemmas/seed-{seed}"
    my_classifiers["mlp_indicator-lemmas"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=cv_lemmas[run],
            outcome_space=outcome_space,
            encoder=token_ind_encoder,
            model_option="mlp",
            save_to=run_folder,
            smlp_args={
                "hidden_dim": 128,
                "dropout": 0.1,
                "batch_size": 512,
                "learning_rate": 1e-4,
                "weight_decay": 1e-4,
                "num_epochs": 25,
            },
        )
    )

accuracies = my_classifiers["mlp_indicator-lemmas"][0]["accuracies"]

for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

  0%|          | 0/5 [00:00<?, ?it/s]

Transforming documents...
Document transformation done! Time taken: 0.584 seconds
Dataloader done! Time taken: 2.547 seconds
Training...


  0%|          | 0/25 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.6667 | train acc: 0.6149 | val loss: 0.6520 | val acc: 0.6189
Epoch 02 | train loss: 0.6363 | train acc: 0.6265 | val loss: 0.6280 | val acc: 0.6330
Epoch 03 | train loss: 0.6095 | train acc: 0.6544 | val loss: 0.6098 | val acc: 0.6548
Epoch 04 | train loss: 0.5884 | train acc: 0.6762 | val loss: 0.5985 | val acc: 0.6709
Epoch 05 | train loss: 0.5730 | train acc: 0.6914 | val loss: 0.5927 | val acc: 0.6747
Epoch 06 | train loss: 0.5617 | train acc: 0.7008 | val loss: 0.5899 | val acc: 0.6765
Epoch 07 | train loss: 0.5532 | train acc: 0.7073 | val loss: 0.5884 | val acc: 0.6783
Epoch 08 | train loss: 0.5461 | train acc: 0.7129 | val loss: 0.5882 | val acc: 0.6785


In [ ]:
evaluate_on_test("mlp_indicator-lemmas", test_lemmas, token_ind_encoder, "mlp");

Exp 2.B: MLP with Bag of Words

In [ ]:
my_classifiers["mlp_bow-lemmas"] = []

for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/mlp_bow-lemmas/seed-{seed}"
    my_classifiers["mlp_bow-lemmas"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=cv_lemmas[run],
            outcome_space=outcome_space,
            encoder=bow_encoder,
            model_option="mlp",
            save_to=run_folder,
            smlp_args={
                "hidden_dim": 128,
                "dropout": 0.1,
                "batch_size": 512,
                "learning_rate": 1e-4,
                "weight_decay": 1e-4,
                "num_epochs": 25,
            },
        )
    )

accuracies = my_classifiers["mlp_bow-lemmas"][0]["accuracies"]

for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

In [ ]:
evaluate_on_test("mlp_bow-lemmas", test_lemmas, bow_encoder, "mlp");

Exp 2.C: MLP with Sentence Embeddings

In [ ]:
my_classifiers["mlp_embeddings"] = []

for run, seed in enumerate(my_seeds):
    run_folder = f"{results_folder}/mlp_embeddings/seed-{seed}"
    my_classifiers["mlp_embeddings"].append(
        part1.experiment.cross_validation_experiment(
            data_splits=cv_sentences[run],
            outcome_space=outcome_space,
            encoder=sentence_em_encoder,
            model_option="mlp",
            save_to=run_folder,
            smlp_args={
                "hidden_dim": 128,
                "dropout": 0.1,
                "batch_size": 512,
                "learning_rate": 1e-3,
                "weight_decay": 1e-4,
                "num_epochs": 25,
            },
        )
    )

accuracies = my_classifiers["mlp_embeddings"][0]["accuracies"]

for fold, acc in accuracies.items():
    print(f"fold {fold}: accuracy {acc:.4f}")

In [ ]:
evaluate_on_test("mlp_embeddings", test_sentences, sentence_em_encoder, "mlp");

In [ ]:
# summary: test accuracy and macro F1 for every experiment (mean and std over the fold models)
summary = pd.DataFrame({
    name: {
        "accuracy": scores.accuracy.mean(), "accuracy_std": scores.accuracy.std(),
        "f1": scores.f1.mean(), "f1_std": scores.f1.std(),
    }
    for name, scores in test_scores.items()
}).T
majority = test_df[LABEL_COL].value_counts(normalize=True).max()
print(f"majority-class baseline: accuracy {majority:.4f}, macro F1 {majority / (1 + majority):.4f}")
summary.round(4)